<a href="https://colab.research.google.com/github/subiksha0515/Recommendation_of_RAG/blob/main/Context_Compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Project Title : **Query-Focused RAG Reader**

---

## 📖 Notebook Overview

This notebook demonstrates an advanced Retrieval-Augmented Generation (RAG) workflow that reads a long academic PDF, retrieves relevant content using embeddings and cosine similarity, and then applies contextual compression to present only the query-focused information in a clear and readable form.

The notebook shows the complete pipeline from document loading to user interaction through a Gradio interface, while clearly comparing the output **before** and **after** compression.

---

## 🎯 What This Notebook Helps to Understand

This notebook helps to understand:

* How corrupted PDF text affects retrieval and how to fix it
* How intelligent chunking improves embedding quality
* How embeddings convert text into vectors for semantic search
* How cosine similarity retrieves meaningful content
* Why normal retrieval returns large unreadable chunks
* How contextual compression improves readability
* How to build a complete document QA system using open-source tools

---

## 🧠 Models and Techniques Used in Each Stage

| Stage         | Model / Technique Used                 | Purpose                          |
| ------------- | -------------------------------------- | -------------------------------- |
| PDF Loading   | PyPDFLoader                            | Extract text from document       |
| Text Cleaning | Regex normalization                    | Fix corrupted PDF text           |
| Chunking      | RecursiveCharacterTextSplitter         | Preserve context while splitting |
| Embeddings    | SentenceTransformer (all-MiniLM-L6-v2) | Convert text into vectors        |
| Retrieval     | FAISS + Cosine Similarity              | Find semantically similar chunks |
| Compression   | FLAN-T5 (open LLM)                     | Extract only relevant lines      |
| Interface     | Gradio UI                              | Visual comparison of results     |

---

## ⭐ Key Takeaway

This notebook shows that **retrieval alone is not enough** in RAG systems. By combining cosine similarity with contextual compression, large academic documents can be transformed into concise, user-friendly answers that significantly improve readability and understanding.


✅ Cell 1 — Install Libraries

In [ ]:
!pip install -q langchain langchain-community langchain-core faiss-cpu pypdf sentence-transformers gradio transformers


✅ Cell 2 — Imports

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import gradio as gr


In [ ]:
!pip show langchain

Name: langchain
Version: 1.2.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


✅ Cell 3 — Connect Google Drive (easy document loading)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


✅ Cell 4 — Load PDF

In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/content/drive/MyDrive/JOBMARKET-REPORT.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()


def fix_pdf_text(text):
    # 1) Fix broken letters: "W o r d" → "Word"
    text = re.sub(r'(?<=\w)\s+(?=\w)', '', text)

    # 2) Add space between lowercase and uppercase: "systemExtracts" → "system Extracts"
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    # 3) Add space between letters and numbers: "Chapter3" → "Chapter 3"
    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)

    # 4) Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# Apply fix to every page
for d in docs:
    d.page_content = fix_pdf_text(d.page_content)

print("Total pages loaded and cleaned:", len(docs))


Total pages loaded and cleaned: 28


| ❌ Raw Text Extracted from PDF (Before Fix)                | ✅ Clean Text After `fix_pdf_text()` (After Fix)                    | Issue Type Fixed                     |
| --------------------------------------------------------- | ------------------------------------------------------------------ | ------------------------------------ |
| `W o r d 2 V e c o r T F - I D F e m b e d d i n g s`     | `Word2Vec or TF-IDF embeddings`                                    | Broken letters separated by spaces   |
| `thesystemextractskeyentitiesuchasName,Education,Skills`  | `the system extracts key entities such as Name, Education, Skills` | Words glued together without spaces  |
| `CHAPTER3OBJECTIVESANDMETHODOLOGY3.1MainObjective`        | `CHAPTER 3 OBJECTIVES AND METHODOLOGY 3.1 Main Objective`          | Letters and numbers stuck together   |
| `skillgapAnalysisModuleusesSemanticSimilarity`            | `skill gap Analysis Module uses Semantic Similarity`               | Lowercase and Uppercase words merged |
| `The     system     performs     NLP     parsing`         | `The system performs NLP parsing`                                  | Multiple unnecessary spaces          |
| `c a n d i d a t e ' s s k i l l s a r e c o m p a r e d` | `candidate's skills are compared`                                  | Character-level spacing issue        |


✅ Cell 5 — Recursive Character Text Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)
print("Total chunks created:", len(chunks))


Total chunks created: 60


✅ Cell 6 — Open-Source Embedding Model (no OpenAI)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed(texts):
    return model.encode(texts, normalize_embeddings=True)

texts = [c.page_content for c in chunks]
embeddings = embed(texts)


✅ Cell 7 — FAISS + Cosine Similarity

In [ ]:
# Rebuild FAISS index safely
texts = [c.page_content for c in chunks]
embeddings = embed(texts)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # cosine similarity
index.add(np.array(embeddings))

def retrieve(query, k=3):
    q_emb = embed([query])
    scores, indices = index.search(np.array(q_emb), k)

    results = []
    for i in indices[0]:
        if i < len(texts):   # safety check
            results.append(texts[i])

    return results


✅ Cell 8 — BEFORE Compression

In [ ]:
def get_before(query):
    results = retrieve(query)
    return "\n\n".join(results)


✅ Cell 9 — Contextual Compression using open LLM (FLAN-T5)

In [ ]:
from transformers import pipeline

compressor = pipeline("text2text-generation", model="google/flan-t5-base")

def compress_text(query, text):
    prompt = f"""
    Extract only the lines relevant to the question and return as a proper paragraph.

    Question: {query}
    Text: {text}
    """

    out = compressor(
        prompt,
        max_length=200,
        do_sample=False,
        num_beams=4,
        early_stopping=True
    )

    return out[0]['generated_text']



Device set to use cpu


In [ ]:
def clean_text(text):
    # remove unnecessary line breaks
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text


✅ Cell 10 — AFTER Compression

In [ ]:
def get_after(query):
    before = get_before(query)
    raw = compress_text(query, before)
    return clean_text(raw)


✅ Cell 11 — Print Before & After (very important demo)

In [ ]:
query = "How does the system perform skill gap analysis?"

before_text = get_before(query)
after_text = get_after(query)

before_len = len(before_text)
after_len = len(after_text)

print("🔴 BEFORE COMPRESSION")
print("-" * 60)
# print(before_text)   # paragraph view
print(f"Character count : {before_len}")
print(f"Word count      : {len(before_text.split())}")




print("\n\n🟢 AFTER COMPRESSION")
print("-" * 60)
print(f"Character count : {after_len}")
print(f"Word count      : {len(after_text.split())}")
print()
print(after_text)           # clean paragraph


print("\n\n📊 COMPRESSION SUMMARY")
print("-" * 60)
reduction = ((before_len - after_len) / before_len) * 100
print(f"Content reduced by : {reduction:.2f}%")


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔴 BEFORE COMPRESSION
------------------------------------------------------------
Character count : 1895
Word count      : 98


🟢 AFTER COMPRESSION
------------------------------------------------------------
Character count : 305
Word count      : 43

Following entity extraction, skill vectors are created using Word 2 Vecand TF-IDFembeddings. AGap Index near 0 indicates strong skill alignment; values > 0.5 suggest missing or outdated competencies (Mikolovetal., 2013). NLP integration to make career guidance accessible to all categories of workers [7].


📊 COMPRESSION SUMMARY
------------------------------------------------------------
Content reduced by : 83.91%


✅ Cell 12 — Gradio UI (final showcase)

In [ ]:
def demo(query):
    before = get_before(query)
    after = get_after(query)
    return before, after


with gr.Blocks() as demo_ui:
    gr.Markdown("# 📄 Query-Focused Document Reader")
    gr.Markdown("Compare cosine similarity retrieval vs contextual compression")

    query = gr.Textbox(label="Enter your question", lines=2)
    submit_btn = gr.Button("Submit Query")

    with gr.Row():
        before_box = gr.Textbox(label="🔴 BEFORE Compression",
                                lines=25,
                                max_lines=30)

        after_box = gr.Textbox(label="🟢 AFTER Compression",
                               lines=25,
                               max_lines=30)

    submit_btn.click(demo, inputs=query, outputs=[before_box, after_box])

demo_ui.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01aa92eaababc374cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




## ✅ Final Conclusion

This project presents a smart document reader that improves traditional RAG by combining cosine similarity retrieval with contextual compression. Instead of showing long academic paragraphs, the system extracts only the lines relevant to the user’s query and presents them in a clear, readable form.



## 🌍 Applications & Use Cases

This approach is useful for:

* Research paper and thesis reading
* Legal and government document analysis
* Technical documentation understanding
* Academic report summarization
* Resume and job description analysis
* Enterprise knowledge base search systems



## 🎯 Outcomes of the Project

* Cleaned corrupted PDF text for accurate processing
* Context-preserving chunking for better embeddings
* Semantic retrieval using cosine similarity
* Significant reduction of content size after compression
* Clear Before vs After comparison for user understanding
* Fully open-source implementation with interactive Gradio UI



